# GEPA Prompt Evolution

**Genetic Pareto Algorithm for prompt optimization**

This notebook evolves prompt templates for a given category (sentiment, NER, factual, etc.)
using a small GGUF model on Colab's free T4 GPU.

### Workflow:
1. Install dependencies + mount Google Drive
2. Download a GGUF model
3. Upload your eval dataset (JSON)
4. Configure the evolution
5. Run GEPA evolution
6. Review results saved to Google Drive

---

In [ ]:
# ============================================================
# CELL 1: Setup — Install deps, mount Drive, create dirs
# ============================================================
import sys
import subprocess
import os
from pathlib import Path

print("=" * 60)
print("GEPA Evolution — Setup")
print("=" * 60)

# --- Install llama-cpp-python with CUDA support ---
print("\n[1/4] Installing llama-cpp-python (CUDA support)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "llama-cpp-python",
    "--extra-index-url",
    "https://abetlen.github.io/llama-cpp-python/whl/cu124",
    "-q"
])
print("  \u2713 llama-cpp-python installed")

# --- Install other deps ---
print("\n[2/4] Installing other dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "vaderSentiment",
    "tqdm",
    "huggingface_hub",
    "requests",
    "-q"
])
print("  \u2713 vaderSentiment, tqdm, huggingface_hub, requests installed")

# --- Mount Google Drive ---
print("\n[3/4] Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("  \u2713 Drive mounted")

# --- Create output directories ---
print("\n[4/4] Creating output directories...")
DRIVE_BASE = Path("/content/drive/MyDrive/GEPA_Results")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
(DRIVE_BASE / "models").mkdir(exist_ok=True)
(DRIVE_BASE / "snapshots").mkdir(exist_ok=True)
(DRIVE_BASE / "reports").mkdir(exist_ok=True)
print(f"  \u2713 Output directory: {DRIVE_BASE}")

print("\n" + "=" * 60)
print("Setup complete!")
print("=" * 60)

## Download GGUF Model

Select a model to download. These are small (~1B param) Q4_K_M quantized models
that fit comfortably in Colab's T4 GPU memory (~16GB VRAM).

In [ ]:
# ============================================================
# CELL 2: Model Download — Download GGUF from Hugging Face
# ============================================================
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

print("=" * 60)
print("Model Download")
print("=" * 60)

# --- Model registry ---
AVAILABLE_MODELS = {
    "qwen2.5-1.5b-instruct-q4_k_m.gguf": {
        "repo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
        "file": "qwen2.5-1.5b-instruct-q4_k_m.gguf",
        "size_gb": 0.97,
        "description": "Qwen 2.5 1.5B Instruct — Good general purpose"
    },
    "qwen2.5-coder-1.5b-q4_k_m.gguf": {
        "repo": "Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF",
        "file": "qwen2.5-coder-1.5b-instruct-q4_k_m.gguf",
        "size_gb": 0.97,
        "description": "Qwen 2.5 Coder 1.5B — Good for structured outputs"
    },
    "gemma-3-1b-it-q4_k_m.gguf": {
        "repo": "google/gemma-3-1b-it-GGUF",
        "file": "gemma-3-1b-it-q4_k_m.gguf",
        "size_gb": 0.78,
        "description": "Gemma 3 1B IT — Google's lightweight instruct model"
    }
}

MODEL_CACHE_DIR = DRIVE_BASE / "models"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("\nAvailable models:")
for i, (key, info) in enumerate(AVAILABLE_MODELS.items(), 1):
    print(f"  {i}. {key}")
    print(f"     {info['description']} ({info['size_gb']} GB)")

# --- Let user pick ---
print("\n" + "-" * 40)
choices = list(AVAILABLE_MODELS.keys())
print(f"Enter model number (1-{len(choices)}) or full filename to download:")
user_input = input("Model choice: ").strip()

# Resolve selection
if user_input.isdigit() and 1 <= int(user_input) <= len(choices):
    model_key = choices[int(user_input) - 1]
elif user_input in AVAILABLE_MODELS:
    model_key = user_input
else:
    # Default to first
    print(f"  Unrecognized '{user_input}', defaulting to {choices[0]}")
    model_key = choices[0]

model_info = AVAILABLE_MODELS[model_key]
local_path = MODEL_CACHE_DIR / model_key

print(f"\nSelected: {model_key}")
print(f"From: {model_info['repo']}")

# --- Download if not cached ---
if local_path.exists():
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f"  \u2713 Already cached ({size_mb:.0f} MB)")
else:
    print(f"  Downloading... (this may take a few minutes)")
    downloaded_path = hf_hub_download(
        repo_id=model_info["repo"],
        filename=model_info["file"],
        local_dir=MODEL_CACHE_DIR,
        local_dir_use_symlinks=False,
        resume_download=True,
    )
    # Rename if needed
    if downloaded_path != str(local_path):
        import shutil
        shutil.move(downloaded_path, local_path)
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f"  \u2713 Downloaded ({size_mb:.0f} MB)")

print(f"\nModel ready at: {local_path}")
print("=" * 60)

## Upload Eval Dataset

Provide a JSON file of evaluation questions in this format:
```json
[
  {"category": "sentiment", "prompt": "I love this!", "expected_answer": "positive"},
  {"category": "ner", "prompt": "John works at Google.", "expected_answer": "[\"John\", \"Google\"]"},
  ...
]
```

You can either upload a file OR provide a public URL.

In [ ]:
# ============================================================
# CELL 3: Eval Dataset Upload
# ============================================================
import json
from collections import Counter
from pathlib import Path

print("=" * 60)
print("Eval Dataset Upload")
print("=" * 60)

eval_data = None

# --- Option 1: Upload file ---
print("\n[Option 1] Upload a JSON file")
print("  Click the folder icon -> Upload, then enter the path below")
upload_path = input("Uploaded file path (or press Enter to skip): ").strip()

if upload_path and Path(upload_path).exists():
    with open(upload_path) as f:
        eval_data = json.load(f)
    print(f"  \u2713 Loaded {len(eval_data)} questions from uploaded file")

# --- Option 2: URL ---
if eval_data is None:
    print("\n[Option 2] Load from URL")
    url = input("Public JSON URL (or press Enter to use demo data): ").strip()
    if url:
        import requests
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            eval_data = resp.json()
            print(f"  \u2713 Loaded {len(eval_data)} questions from URL")
        except Exception as e:
            print(f"  \u2717 Error loading URL: {e}")

# --- Option 3: Demo data ---
if eval_data is None:
    print("\n[Option 3] Using demo sentiment dataset")
    eval_data = [
        {"category": "sentiment", "prompt": "This product is amazing!", "expected_answer": "positive"},
        {"category": "sentiment", "prompt": "I hate this so much.", "expected_answer": "negative"},
        {"category": "sentiment", "prompt": "The movie was okay, not great.", "expected_answer": "neutral"},
        {"category": "sentiment", "prompt": "Absolutely wonderful experience!", "expected_answer": "positive"},
        {"category": "sentiment", "prompt": "Worst purchase ever.", "expected_answer": "negative"},
        {"category": "sentiment", "prompt": "It works as expected.", "expected_answer": "neutral"},
        {"category": "sentiment", "prompt": "I'm thrilled with the results!", "expected_answer": "positive"},
        {"category": "sentiment", "prompt": "Terrible customer support.", "expected_answer": "negative"},
        {"category": "sentiment", "prompt": "Not bad, could be better.", "expected_answer": "neutral"},
        {"category": "sentiment", "prompt": "Best thing I've bought all year!", "expected_answer": "positive"},
        {"category": "sentiment", "prompt": "Completely useless product.", "expected_answer": "negative"},
        {"category": "sentiment", "prompt": "Does the job, nothing special.", "expected_answer": "neutral"},
    ]
    print(f"  \u2713 Using {len(eval_data)} demo questions")

# --- Validate and show stats ---
print("\n" + "-" * 40)
print("Dataset Stats:")
print(f"  Total questions: {len(eval_data)}")

categories = Counter(item.get("category", "unknown") for item in eval_data)
print(f"  Categories: {dict(categories)}")

# Detect primary category
primary_category = categories.most_common(1)[0][0]
print(f"  Primary category: {primary_category}")

# Save to Drive for later use
eval_path = DRIVE_BASE / "eval_data.json"
with open(eval_path, "w") as f:
    json.dump(eval_data, f, indent=2)
print(f"  Saved to: {eval_path}")

print("\n" + "=" * 60)
print("Eval data ready!")
print("=" * 60)

## GEPA Engine

This cell implements the full Genetic Pareto Algorithm:
- **Seed generation**: Creates prompt templates from base patterns
- **Evaluation**: Runs each prompt against the eval dataset using the GGUF model
- **Scoring**: 4-cascade fuzzy matching (exact → substring → numeric 1% → token overlap 80%)
- **Selection**: Picks elite prompts by accuracy
- **Mutation**: Generates new prompts via synonym swap, constraint addition, temperature change
- **Pareto tracking**: Monitors (accuracy, token count, latency) trade-offs

In [ ]:
# ============================================================
# CELL 4: GEPA Engine — Full implementation
# ============================================================
import json
import re
import time
import random
import math
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import Counter

import numpy as np
from tqdm.notebook import tqdm

print("=" * 60)
print("GEPA Engine — Loading implementation")
print("=" * 60)

# -----------------------------------------------------------
# Fuzzy Matching (4-cascade)
# -----------------------------------------------------------

def tokenize(text: str) -> set:
    """Extract lowercase alphanumeric tokens."""
    return set(re.findall(r'[a-zA-Z0-9]+', text.lower()))


def fuzzy_match(answer: str, expected: str) -> float:
    """
    4-cascade fuzzy matching. Returns 1.0 (full), 0.75, 0.5, 0.25, or 0.0.
    Cascade: exact -> substring (if expected <= 20 chars) -> numeric 1% -> token overlap 80%
    """
    a = answer.strip()
    e = expected.strip()

    # Cascade 1: Exact match (case-insensitive)
    if a.lower() == e.lower():
        return 1.0

    # Cascade 2: Substring match (if expected is short, <= 20 chars)
    if len(e) <= 20 and e.lower() in a.lower():
        return 0.75

    # Cascade 3: Numeric tolerance (1%)
    try:
        a_num = float(a)
        e_num = float(e)
        if e_num != 0 and abs(a_num - e_num) / abs(e_num) <= 0.01:
            return 0.5
        if e_num == 0 and abs(a_num) <= 0.01:
            return 0.5
    except (ValueError, TypeError):
        pass

    # Cascade 4: Token overlap >= 80%
    a_tokens = tokenize(a)
    e_tokens = tokenize(e)
    if e_tokens and a_tokens:
        overlap = len(a_tokens & e_tokens) / len(e_tokens)
        if overlap >= 0.8:
            return 0.25

    return 0.0


# -----------------------------------------------------------
# Prompt Template Representation
# -----------------------------------------------------------

@dataclass
class PromptCell:
    """A single prompt template with its evaluation results."""
    system_prompt: str
    user_template: str
    temperature: float = 0.0
    max_tokens: int = 64
    generation: int = 0
    cell_id: str = ""

    # Evaluation results
    accuracy: float = 0.0
    avg_tokens: float = 0.0
    avg_latency: float = 0.0
    pareto_rank: int = 0

    def to_dict(self) -> dict:
        return {
            "cell_id": self.cell_id,
            "generation": self.generation,
            "system_prompt": self.system_prompt,
            "user_template": self.user_template,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "accuracy": self.accuracy,
            "avg_tokens": self.avg_tokens,
            "avg_latency": self.avg_latency,
        }


# -----------------------------------------------------------
# Prompt Templates / Mutations
# -----------------------------------------------------------

SYSTEM_TEMPLATES = [
    "You are a precise {category} classifier. Answer with exactly one word.",
    "Classify the following text by {category}. Respond with only the category label.",
    "You are an expert at {category} analysis. Give a concise, accurate answer.",
    "Analyze the {category} of this text. Output just the classification label.",
    "You are a {category} detection system. Provide only the classification result.",
]

USER_TEMPLATES = [
    "Text: {prompt}\n{category}:",
    "Input: {prompt}\nWhat is the {category}?\nAnswer:",
    "{prompt}\n\nClassify the {category} above:",
    "Determine the {category} of the following:\n{prompt}\n\nAnswer:",
    "{prompt}\n\n{category} classification:",
]

SYNONYM_MAP = {
    "classify": ["categorize", "identify", "determine", "analyze", "evaluate"],
    "analyze": ["examine", "assess", "process", "evaluate", "interpret"],
    "precise": ["accurate", "exact", "concise", "specific", "clear"],
    "answer": ["response", "output", "result", "label", "prediction"],
    "text": ["input", "content", "passage", "statement", "sentence"],
    "expert": ["specialist", "system", "model", "agent", "assistant"],
    "provide": ["output", "return", "give", "produce", "generate"],
    "label": ["tag", "class", "category", "type", "classification"],
}

CONSTRAINT_PHRASES = [
    "",
    " Be concise.",
    " Use only lowercase.",
    " Do not explain.",
    " Answer in one word.",
    " Be precise.",
    " No punctuation.",
]


def create_seed_cells(category: str, count: int, gen: int = 0) -> List[PromptCell]:
    """Create seed PromptCells from template combinations."""
    cells = []
    for i in range(count):
        sys_t = random.choice(SYSTEM_TEMPLATES)
        usr_t = random.choice(USER_TEMPLATES)
        temp = 0.0 if random.random() < 0.7 else round(random.uniform(0.1, 0.5), 2)
        cell = PromptCell(
            system_prompt=sys_t.format(category=category),
            user_template=usr_t,
            temperature=temp,
            generation=gen,
            cell_id=f"seed_{gen}_{i}",
        )
        cells.append(cell)
    return cells


def mutate_cell(cell: PromptCell, category: str) -> PromptCell:
    """Mutate a PromptCell to create a new variant."""
    new_sys = cell.system_prompt
    new_usr = cell.user_template
    new_temp = cell.temperature

    mutations = ["synonym", "constraint", "template_swap", "temperature"]
    mutation = random.choice(mutations)

    if mutation == "synonym":
        # Replace a synonym in the system prompt
        for word, syns in SYNONYM_MAP.items():
            if word in new_sys.lower():
                new_sys = re.sub(
                    rf'\b{word}\b',
                    random.choice(syns),
                    new_sys,
                    flags=re.IGNORECASE
                )
                break

    elif mutation == "constraint":
        # Add or swap a constraint phrase
        constraint = random.choice(CONSTRAINT_PHRASES)
        if constraint and constraint not in new_sys:
            new_sys += constraint

    elif mutation == "template_swap":
        # Replace user template with a random one
        new_usr = random.choice(USER_TEMPLATES)

    elif mutation == "temperature":
        # Adjust temperature
        new_temp = round(random.uniform(0.0, 0.5), 2)

    return PromptCell(
        system_prompt=new_sys.format(category=category)
            if "{category}" in new_sys else new_sys,
        user_template=new_usr,
        temperature=new_temp,
        generation=cell.generation + 1,
        cell_id=f"mut_{cell.generation + 1}_{random.randint(0, 99999)}",
    )


# -----------------------------------------------------------
# Pareto Dominance
# -----------------------------------------------------------

def dominates(a: PromptCell, b: PromptCell) -> bool:
    """Check if cell a Pareto-dominates cell b.
    We maximize accuracy, minimize tokens, minimize latency."""
    acc_a = a.accuracy >= b.accuracy
    tok_a = a.avg_tokens <= b.avg_tokens
    lat_a = a.avg_latency <= b.avg_latency
    # Strictly better in at least one
    strictly = (a.accuracy > b.accuracy or
                a.avg_tokens < b.avg_tokens or
                a.avg_latency < b.avg_latency)
    return (acc_a and tok_a and lat_a) and strictly


def compute_pareto_front(cells: List[PromptCell]) -> List[PromptCell]:
    """Return the non-dominated cells (Pareto front)."""
    front = []
    for cell in cells:
        dominated = False
        for other in cells:
            if other is not cell and dominates(other, cell):
                dominated = True
                break
        if not dominated:
            front.append(cell)
    return front


# -----------------------------------------------------------
# Model Loader (unloads after each evaluation to free VRAM)
# -----------------------------------------------------------

def load_model(model_path: str, n_gpu_layers: int = -1, n_ctx: int = 2048):
    """Load a GGUF model with CUDA offloading."""
    from llama_cpp import Llama
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=n_gpu_layers,
        n_ctx=n_ctx,
        verbose=False,
    )
    return llm


def unload_model(llm):
    """Unload model to free VRAM."""
    if llm is not None:
        import gc
        del llm
        gc.collect()
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# -----------------------------------------------------------
# Evaluation Runner
# -----------------------------------------------------------

def evaluate_cell(
    cell: PromptCell,
    eval_data: List[Dict],
    llm,
) -> Tuple[float, float, float]:
    """
    Evaluate a PromptCell across all eval questions.
    Returns (accuracy, avg_tokens, avg_latency).
    """
    scores = []
    token_counts = []
    latencies = []

    for item in eval_data:
        prompt_text = item.get("prompt", "")
        expected = item.get("expected_answer", "")

        # Build user message from template
        user_msg = cell.user_template.format(
            prompt=prompt_text,
            category=item.get("category", "unknown")
        )

        # Call model with temperature=0.0 for deterministic eval
        t0 = time.time()
        try:
            response = llm.create_chat_completion(
                messages=[
                    {"role": "system", "content": cell.system_prompt},
                    {"role": "user", "content": user_msg},
                ],
                temperature=cell.temperature,
                max_tokens=cell.max_tokens,
            )
            latency = time.time() - t0
            answer = response["choices"][0]["message"]["content"].strip()
            tokens_used = response.get("usage", {}).get("completion_tokens", len(answer) // 4)
        except Exception as e:
            print(f"  [WARN] LLM call failed: {e}")
            answer = ""
            latency = time.time() - t0
            tokens_used = 0

        score = fuzzy_match(answer, expected)
        scores.append(score)
        token_counts.append(tokens_used)
        latencies.append(latency)

    accuracy = np.mean(scores) if scores else 0.0
    avg_tokens = np.mean(token_counts) if token_counts else 0.0
    avg_latency = np.mean(latencies) if latencies else 0.0

    return accuracy, avg_tokens, avg_latency


# -----------------------------------------------------------
# Main GEPA Loop
# -----------------------------------------------------------

def run_gepa(
    eval_data: List[Dict],
    model_path: str,
    category: str,
    generations: int = 3,
    population_size: int = 8,
    cells_per_model: int = 4,
    output_dir: Path = DRIVE_BASE,
) -> Dict:
    """
    Run the full GEPA evolution loop.

    Args:
        eval_data: List of {category, prompt, expected_answer} dicts
        model_path: Path to GGUF model file
        category: Target category (e.g. "sentiment", "ner")
        generations: Number of generations to evolve
        population_size: Number of cells per generation
        cells_per_model: How many cells to evaluate before reloading model
        output_dir: Directory for saving results

    Returns:
        Dict with full results
    """
    output_dir = Path(output_dir)

    print(f"\n{'=' * 60}")
    print(f"Starting GEPA Evolution")
    print(f"  Category:       {category}")
    print(f"  Generations:    {generations}")
    print(f"  Population:     {population_size}")
    print(f"  Eval questions: {len(eval_data)}")
    print(f"  Model:          {Path(model_path).name}")
    print(f"{'=' * 60}")

    all_cells = []
    pareto_history = []
    best_overall = None

    for gen in range(generations):
        print(f"\n{'\u2500' * 50}")
        print(f"Generation {gen + 1}/{generations}")
        print(f"{'\u2500' * 50}")

        if gen == 0:
            # Create seed cells
            cells = create_seed_cells(category, population_size, gen)
        else:
            # Create next generation from elites
            cells = []
            # Take top 50% accuracy from previous generation
            prev_sorted = sorted(
                all_cells,
                key=lambda c: c.accuracy,
                reverse=True
            )
            elites = prev_sorted[:max(2, population_size // 2)]

            # Keep elites
            cells.extend(elites)

            # Generate mutants from elites
            while len(cells) < population_size:
                parent = random.choice(elites)
                child = mutate_cell(parent, category)
                child.generation = gen
                cells.append(child)

        # Evaluate each cell
        llm = None
        try:
            pbar = tqdm(cells, desc=f"Eval Gen {gen + 1}", unit="cell")
            for idx, cell in enumerate(pbar):
                # Load model (unload between groups)
                if llm is None:
                    print(f"  Loading model...")
                    llm = load_model(model_path)

                pbar.set_postfix(cell=cell.cell_id)

                acc, toks, lat = evaluate_cell(cell, eval_data, llm)
                cell.accuracy = acc
                cell.avg_tokens = toks
                cell.avg_latency = lat

                # Update best
                if best_overall is None or acc > best_overall.accuracy:
                    best_overall = cell

                # Unload model periodically to manage VRAM
                if (idx + 1) % cells_per_model == 0 and idx + 1 < len(cells):
                    unload_model(llm)
                    llm = None

        finally:
            unload_model(llm)
            llm = None

        all_cells.extend(cells)

        # Compute Pareto front for this generation's cells
        gen_front = compute_pareto_front(cells)
        pareto_history.append({
            "generation": gen,
            "front_cells": [c.to_dict() for c in gen_front],
        })

        # Print generation summary
        print(f"\n  Generation {gen + 1} results:")
        print(f"    Best accuracy:  {max(c.accuracy for c in cells):.4f}")
        print(f"    Avg accuracy:   {np.mean([c.accuracy for c in cells]):.4f}")
        print(f"    Pareto front:   {len(gen_front)} non-dominated cells")

        for c in sorted(cells, key=lambda x: x.accuracy, reverse=True)[:3]:
            print(f"      [{c.cell_id}] acc={c.accuracy:.3f}  tok={c.avg_tokens:.0f}  lat={c.avg_latency:.3f}s")

        # Save snapshot
        snapshot = {
            "generation": gen,
            "cells": [c.to_dict() for c in cells],
            "pareto_front": [c.to_dict() for c in gen_front],
        }
        snap_path = output_dir / "snapshots" / f"gen_{gen}.json"
        with open(snap_path, "w") as f:
            json.dump(snapshot, f, indent=2)
        print(f"  Snapshot saved: {snap_path}")

    # --- Final report ---
    print(f"\n{'=' * 60}")
    print("Evolution Complete!")
    print(f"{'=' * 60}")

    # Overall Pareto front across ALL generations
    final_front = compute_pareto_front(all_cells)

    report = {
        "category": category,
        "model": Path(model_path).name,
        "generations": generations,
        "population_size": population_size,
        "total_cells_evaluated": len(all_cells),
        "best_cell": best_overall.to_dict() if best_overall else None,
        "pareto_front": [c.to_dict() for c in final_front],
        "pareto_history": pareto_history,
        "all_cells": [c.to_dict() for c in all_cells],
    }

    report_path = output_dir / "reports" / "final_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"Final report saved: {report_path}")

    return report


print("\n" + "=" * 60)
print("GEPA Engine loaded and ready!")
print("=" * 60)

## Configuration

Set the evolution parameters below before running.

In [ ]:
# ============================================================
# CELL 5: Configuration
# ============================================================
from pathlib import Path

print("=" * 60)
print("Configuration")
print("=" * 60)

# --- Category ---
# Auto-detect from eval data, or override below
if 'eval_data' in dir() and eval_data:
    from collections import Counter
    cats = Counter(item.get("category", "unknown") for item in eval_data)
    default_category = cats.most_common(1)[0][0]
else:
    default_category = "sentiment"

print(f"\nDetected primary category: {default_category}")
print("Override below if needed:")

# ====== EDIT THESE VALUES ======
CATEGORY = default_category  # e.g. "sentiment", "ner", "factual", "toxicity"

# Model filename (must already be downloaded)
MODEL_NAME = "qwen2.5-1.5b-instruct-q4_k_m.gguf"  # or one of the other downloaded models

# Evolution parameters
GENERATIONS = 3            # Number of generations to run
POPULATION_SIZE = 8        # Cells per generation
CELLS_PER_MODEL = 4        # How many cells to evaluate per model load cycle
# ===============================

# Resolve model path
MODEL_PATH = DRIVE_BASE / "models" / MODEL_NAME

print(f"\n  Category:        {CATEGORY}")
print(f"  Model:           {MODEL_NAME}")
print(f"  Model path:      {MODEL_PATH}")
print(f"  Model exists:    {MODEL_PATH.exists()}")
print(f"  Generations:     {GENERATIONS}")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Cells/model:     {CELLS_PER_MODEL}")

if not MODEL_PATH.exists():
    print(f"\n  \u26a0 Model file not found! Re-run Cell 2 to download it.")
else:
    size_gb = Path(MODEL_PATH).stat().st_size / (1024**3)
    print(f"  Model size:      {size_gb:.2f} GB")

print("\n" + "=" * 60)
print("Configuration done — proceed to Cell 6 to run!")
print("=" * 60)

## Run GEPA Evolution

Execute the evolution loop. This will:
1. Create seed prompts for your category
2. Evaluate each against your dataset using the GGUF model
3. Select elites and mutate to create next generation
4. Track Pareto front across all generations
5. Save snapshots and final report to Google Drive

⏱ **Estimated time**: ~3-5 minutes per generation with a 1.5B model on T4.

In [ ]:
# ============================================================
# CELL 6: Run GEPA Evolution
# ============================================================
import time

print("=" * 60)
print("Executing GEPA Evolution")
print("=" * 60)

# Validate inputs
if 'eval_data' not in dir() or not eval_data:
    raise RuntimeError("No eval data loaded! Run Cell 3 first.")

if not MODEL_PATH.exists():
    raise RuntimeError(f"Model not found at {MODEL_PATH}. Run Cell 2 to download.")

print(f"\nStarting at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
t_start = time.time()

# Run GEPA
report = run_gepa(
    eval_data=eval_data,
    model_path=str(MODEL_PATH),
    category=CATEGORY,
    generations=GENERATIONS,
    population_size=POPULATION_SIZE,
    cells_per_model=CELLS_PER_MODEL,
    output_dir=DRIVE_BASE,
)

t_elapsed = time.time() - t_start
print(f"\n{'=' * 60}")
print(f"Evolution finished in {t_elapsed:.1f}s ({t_elapsed / 60:.1f} min)")
print(f"{'=' * 60}")

## Results

View the best cells, Pareto front, and download the full report.

In [ ]:
# ============================================================
# CELL 7: Results
# ============================================================
import json
from pathlib import Path
from datetime import datetime
import subprocess
import os

print("=" * 60)
print("GEPA Evolution Results")
print("=" * 60)

if 'report' not in dir() or report is None:
    # Try loading from Drive
    report_path = DRIVE_BASE / "reports" / "final_report.json"
    if report_path.exists():
        with open(report_path) as f:
            report = json.load(f)
        print("Loaded report from Drive backup.\n")
    else:
        print("No report found \u2014 run Cell 6 first!")
        report = {}

# --- Best Cell ---
print("\n" + "\u2500" * 50)
print("\U0001f947 Best Overall Cell")
print("\u2500" * 50)
if report.get("best_cell"):
    bc = report["best_cell"]
    print(f"  Cell ID:       {bc['cell_id']}")
    print(f"  Generation:    {bc['generation']}")
    print(f"  Accuracy:      {bc['accuracy']:.4f}")
    print(f"  Avg Tokens:    {bc['avg_tokens']:.1f}")
    print(f"  Avg Latency:   {bc['avg_latency']:.3f}s")
    print(f"  Temperature:   {bc['temperature']}")
    print(f"  System Prompt: {bc['system_prompt']}")
    print(f"  User Template: {bc['user_template']}")
else:
    print("  No best cell found.")

# --- Best per generation ---
print("\n" + "\u2500" * 50)
print("\U0001f4ca Best Cell Per Generation")
print("\u2500" * 50)
if report.get("all_cells"):
    by_gen = {}
    for c in report["all_cells"]:
        g = c["generation"]
        if g not in by_gen or c["accuracy"] > by_gen[g]["accuracy"]:
            by_gen[g] = c
    for g in sorted(by_gen.keys()):
        c = by_gen[g]
        print(f"  Gen {g}: acc={c['accuracy']:.3f}  tok={c['avg_tokens']:.0f}  lat={c['avg_latency']:.3f}s  [{c['cell_id']}]")

# --- Pareto Front ---
print("\n" + "\u2500" * 50)
print("\U0001f3af Pareto Front (non-dominated cells)")
print("\u2500" * 50)
if report.get("pareto_front"):
    print(f"  {'Cell ID':<20} {'Accuracy':<10} {'Tokens':<8} {'Latency':<8}")
    print(f"  {'-'*20} {'-'*10} {'-'*8} {'-'*8}")
    for c in sorted(report["pareto_front"], key=lambda x: -x["accuracy"]):
        print(f"  {c['cell_id']:<20} {c['accuracy']:<10.3f} {c['avg_tokens']:<8.0f} {c['avg_latency']:<8.3f}")
else:
    print("  No Pareto front data.")

# --- All cells table ---
print("\n" + "\u2500" * 50)
print("\U0001f4cb All Evaluated Cells")
print("\u2500" * 50)
if report.get("all_cells"):
    print(f"  {'Cell ID':<20} {'Gen':<4} {'Accuracy':<10} {'Tokens':<8} {'Latency':<8}")
    print(f"  {'-'*20} {'-'*4} {'-'*10} {'-'*8} {'-'*8}")
    for c in sorted(report["all_cells"], key=lambda x: -x["accuracy"]):
        print(f"  {c['cell_id']:<20} {c['generation']:<4} {c['accuracy']:<10.3f} {c['avg_tokens']:<8.0f} {c['avg_latency']:<8.3f}")

# --- Save summary ---
summary_path = DRIVE_BASE / "reports" / "summary.txt"
with open(summary_path, "w") as f:
    f.write(f"GEPA Evolution Results\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Category: {report.get('category', 'N/A')}\n")
    f.write(f"Model: {report.get('model', 'N/A')}\n")
    f.write(f"Total cells evaluated: {report.get('total_cells_evaluated', 0)}\n\n")
    if report.get("best_cell"):
        bc = report["best_cell"]
        f.write(f"Best accuracy: {bc['accuracy']:.4f}\n")
        f.write(f"Best system prompt: {bc['system_prompt']}\n")
        f.write(f"Best user template: {bc['user_template']}\n\n")
    f.write(f"Pareto front size: {len(report.get('pareto_front', []))}\n")
print(f"\n  Summary saved to: {summary_path}")

# --- Download link ---
print("\n" + "\u2500" * 50)
print("\U0001f4be Results on Google Drive")
print("\u2500" * 50)
print(f"  All results:  {DRIVE_BASE}/")
print(f"  Final report: {DRIVE_BASE / 'reports' / 'final_report.json'}")
print(f"  Snapshots:    {DRIVE_BASE / 'snapshots'}/")

# Create a downloadable zip
print("\n  Creating zip for download...")
zip_path = "/content/GEPA_Results.zip"
subprocess.run(
    ["zip", "-r", zip_path, "/content/drive/MyDrive/GEPA_Results/"],
    capture_output=True, check=False
)
from google.colab import files
if os.path.exists(zip_path):
    files.download(zip_path)
else:
    print("  Zip file not created; results are still on Drive.")

print("\n" + "=" * 60)
print("Done!")
print("=" * 60)

## Troubleshooting

| Issue | Solution |
|-------|----------|
| **Out of memory** | Reduce `POPULATION_SIZE` or `CELLS_PER_MODEL` |
| **Model download fails** | The model files are cached in Drive. Try re-running Cell 2 |
| **CUDA out of memory** | Use a smaller model (gemma-3-1b-it is smallest) |
| **No eval data loaded** | Make sure to run Cell 3 fully before Cell 6 |
| **Session disconnects** | Results are auto-saved to Drive after each generation |

### Notes
- The notebook uses `temperature=0.0` for deterministic evaluation
- Models are loaded/unloaded to manage VRAM carefully
- Each generation's results are snapshotted to Drive immediately